<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

<!-- aviso-traducao-ptbr -->
<sub>
<b>Tradução não oficial para português do Brasil.</b> Este arquivo é uma obra
derivada do repositório original de Sebastian Raschka
(<a href="https://github.com/rasbt/reasoning-from-scratch">rasbt/reasoning-from-scratch</a>),
licenciado sob Apache License 2.0. Apenas o texto foi traduzido; o código
permanece inalterado. Não é uma publicação oficial da Manning e não substitui o
livro. Detalhes das convenções em <code>GLOSSARIO-TRADUCAO.md</code>.
</sub>

# Capítulo 6: Treinando modelos de raciocínio com aprendizado por reforço

Pacotes usados neste notebook:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.16
torch version: 2.10.0
tokenizers version: 0.21.4


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F01_raschka.webp" width=600>

&nbsp;
## 6.1 Introdução ao aprendizado por reforço para LLMs

- O inference-time scaling melhora o raciocínio usando mais computação por resposta gerada
- O training-time scaling melhora o raciocínio usando computação adicional durante o treinamento, que é o foco deste capítulo

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F02_raschka.webp" width=600>

- Inference-time scaling e training-time scaling também podem (/devem) ser combinados, por exemplo, aplicando técnicas de inference-time depois do treinamento de raciocínio baseado em RL
- Na prática, RL para LLMs é aplicado como um estágio de pós-treinamento sobre um modelo pré-treinado ou após o fine-tuning de instruções

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F03_raschka.webp" width=600>

- O pretraining constrói conhecimento geral por predição do próximo token, enquanto o RL refina o comportamento do modelo otimizando objetivos no nível da sequência, como a correção da resposta ou preferências
- RL para LLMs inclui treinamento de raciocínio e ajuste por preferências, mas o RL focado em raciocínio também pode ser aplicado diretamente a um modelo base pré-treinado, como mostrado pelo DeepSeek-R1
- Treinar raciocínio diretamente no modelo base produz um modelo mais fraco, mas ainda capaz (embora ofereça um cenário mais simples para entender o que o estágio de raciocínio contribui)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F04_raschka.webp" width=600>

&nbsp;
### 6.1.1 O pipeline original de aprendizado por reforço com feedback humano (RLHF)

- O RLHF foi introduzido no trabalho do InstructGPT, em 2022, e usa rótulos de preferência humana para treinar LLMs (esse foi um passo decisivo para transformar o GPT-3 no ChatGPT original)
- Diferente do pretraining e do fine-tuning supervisionado, que otimizam a predição do próximo token, o RLHF otimiza modelos com base em rótulos de preferência humana sobre as respostas do modelo

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F05_raschka.webp" width=600>

&nbsp;
### 6.1.2 Do feedback humano aos rewards verificáveis (RLVR)

- O RLHF exige treinar um reward model separado, que costuma ser um LLM grande e caro
- O RLVR substitui o reward model aprendido por rewards determinísticos e verificáveis automaticamente

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F06_raschka.webp" width=600>

- A popularidade do RLVR foi impulsionada em grande parte pelo sucesso do DeepSeek-R1 em 2025, que demonstrou forte desempenho de raciocínio sem depender de dados de preferência humana nem de um reward model aprendido
- O DeepSeek-R1 treinou o comportamento de raciocínio usando rewards verificáveis automaticamente, como checagens de correção para problemas de matemática e compilação ou execução de código para tarefas de programação
- Embora este livro foque em verificação baseada em matemática, a ideia subjacente é parecida com a verificação de código: os rewards são calculados automaticamente usando sinais binários de sucesso

&nbsp;
## 6.2 Passo a passo de aprendizado por reforço com rewards verificáveis usando GRPO

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F07_raschka.webp" width=600>

- Agora, depois de apresentar o panorama geral e ver como o RL se encaixa no ciclo de desenvolvimento de LLMs, vamos implementar RLVR para treinar um modelo de raciocínio (parecido com o DeepSeek-R1-Zero, mas em escala muito menor, já que uma execução comparável custaria várias centenas de milhares de dólares em custos de GPU)
- RL para LLMs usa um chamado algoritmo de policy gradient, usado para atualizar o LLM que queremos treinar (que é chamado de "policy" em contextos de RL)
- Um algoritmo de policy gradient popular para RLHF é o proximal policy optimization (PPO); poderíamos usar o mesmo algoritmo em RLVR
- No entanto, a equipe do DeepSeek usou um algoritmo mais simples ao treinar os modelos de raciocínio DeepSeek-R1, a saber, o group relative policy optimization (GRPO) (usado pela primeira vez no DeepSeekMath)
- O GRPO é mais amigável em termos de recursos porque, no PPO, temos outro LLM calculando a função de valor; no GRPO, não precisamos disso, já que ele deriva seu sinal de aprendizado de comparações relativas dentro de um grupo de respostas amostradas
- Leitores interessados podem encontrar uma comparação lado a lado mais detalhada entre PPO e GRPO no meu artigo [The State of Reinforcement Learning for LLM Reasoning](https://magazine.sebastianraschka.com/p/the-state-of-llm-reasoning-model-training)
- Neste capítulo, implementamos RLVR usando GRPO
- Além disso, o próximo capítulo apresenta melhorias adicionais ao GRPO para melhorar a estabilidade do treinamento e o desempenho de modelagem resultante

### 6.2.1 Intuição de alto nível do GRPO com uma analogia de chef

- Como o GRPO pode parecer complicado à primeira vista, quis começar esta seção com uma visão geral usando uma analogia de "chef e cozinha", para introduzir a terminologia e dar alguma intuição

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F08_raschka.webp" width=600>

- Na maioria dos contextos de RL para LLMs, rollout e completion são termos usados de forma intercambiável

### 6.2.2 O procedimento do GRPO em alto nível

- O roteiro técnico para implementar o GRPO nas seções seguintes:

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F09_raschka.webp" width=600>

&nbsp;
## 6.3 Carregando um modelo pré-treinado

- O código deste capítulo é idêntico ao dos capítulos anteriores

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F10_raschka.webp" width=600>

In [2]:
import torch

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

device = get_device()
device = torch.device("cpu")

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [3]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(0)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

 \boxed{58}

&nbsp;
## 6.4 Carregando um subconjunto de treinamento do MATH

- Usamos um subconjunto de treinamento sem sobreposição, derivado do dataset MATH original, que exclui explicitamente os exemplos do MATH-500 usados para avaliação do modelo nos capítulos anteriores (para mais informações sobre como o dataset foi preparado, veja https://github.com/rasbt/math_full_minus_math500)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F11_raschka.webp" width=600>

- A função `load_math_train` a seguir é parecida com a função [load_math500_test](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch03.py#L422) do capítulo 3, exceto que especificamos um caminho de arquivo diferente

In [4]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)

    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "math_full_minus_math500/refs/heads/main/"
        "math_full_minus_math500.json"
    )
    backup_url = (
        "https://f001.backblazeb2.com/file/reasoning-from-scratch/"
        "MATH/math_full_minus_math500.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
        except requests.RequestException:
            print("Using backup URL.")
            r = requests.get(backup_url, timeout=30)
            r.raise_for_status()

        data = r.json()

        if save_copy:
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

In [5]:
math_train = load_math_train()

print("Dataset size:", len(math_train))

Dataset size: 12000


In [6]:
from pprint import pprint

pprint(math_train[4])

{'answer': '6',
 'level': 'Level 3',
 'problem': 'Sam is hired for a 20-day period. On days that he works, he earns '
            '$\\$$60. For each day that he does not work, $\\$$30 is '
            'subtracted from his earnings. At the end of the 20-day period, he '
            'received $\\$$660. How many days did he not work?',
 'solution': 'Call $x$ the number of days Sam works and $y$ the number of days '
             'he does not. We can set up the following system of equations to '
             'represent the given information: \\begin{align*}\n'
             'x+y &= 20 \\\\\n'
             '60x - 30y &= 660 \\\\\n'
             '\\end{align*} The first equation represents the total number of '
             'days Sam works, and the second equation represents his total '
             'profit. Solving for $x$ in the first equation yields $x = 20 - '
             'y$. Substituting into the second equation gives $60(20-y) - 30y '
             '= 660$. Canceling a factor of $10$ an

- Note que só precisamos dos campos `"answer"` e `"problem"`
- Em tese, pode ser tentador usar o `"solution"`, mas aqui queremos deixar o modelo explorar soluções livremente (em vez de aprender uma solução e um estilo específicos)

&nbsp;
## 6.5 Amostrando rollouts

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F12_raschka.webp" width=600>

- Rollout é jargão de RL para resposta gerada

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F13_raschka.webp" width=400>

- Precisamos do `@torch.no_grad` porque não queremos construir um grafo e retropropagar por isso, mas o `@torch.inference_mode` não funciona (faz coisas demais) e resulta em

> RuntimeError: Inference tensors cannot be saved for backward. Please do not use Tensors created in inference mode in computation tracked by autograd. To work around this, you can make a clone to get a normal tensor and use it in autograd, or use `torch.no_grad()` instead of `torch.inference_mode()`.

In [7]:
from reasoning_from_scratch.qwen3 import KVCache
from reasoning_from_scratch.ch04 import top_p_filter


@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for _ in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        token_id = next_token.item()
        generated.append(token_id)

        if (
            tokenizer.eos_token_id is not None
            and token_id == tokenizer.eos_token_id
        ):
            break
        logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

- Não há nada de novo aqui
- O código acima é simplesmente uma versão mais enxuta do que vínhamos desenvolvendo antes; ele combina a função [generate_text_basic_stream_cache](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/ch02.py#L57) do capítulo 2 com a amostragem por temperature e top-p do capítulo 4, diretamente
- Em vez de emitir cada token, agora apenas coletamos os tokens em um tensor, já que não precisamos imprimir os tokens gerados ao vivo

In [8]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

 \boxed{58}<|endoftext|>


In [9]:
torch.manual_seed(5)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

 Let's solve the problem step by step.

**Given:**
\[
\text{Half the value of } 3x - 9 \text{ is } x + 37.
\]

**Step 1: Translate the statement into an equation.**
\[
\frac{1}{2} (3x - 9) = x + 37
\]

**Step 2: Eliminate the fraction by multiplying both sides by 2.**
\[
3x - 9 = 2(x + 37)
\]

**Step 3: Distribute the 2 on the right side.**
\[
3x - 9 = 2x + 74
\]

**Step 4: Subtract \(2x\) from both sides to get the \(x\)-terms on one side.**
\[
3x - 2x - 9 = 74
\]
\[
x - 9 = 74
\]

**Step 5: Add 9 to both sides to solve for \(x\).**
\[
x = 74 + 9
\]
\[
x = 83
\]

**Final Answer:**
\[
\boxed{83}
\]<|endoftext|>


- Na prática, chamaríamos `sample_response` várias vezes para gerar rollouts
- Para manter o passo a passo do GRPO simples e alinhado com a figura 6.13, assumimos, em vez disso, que o modelo gerou as quatro respostas a seguir:

In [10]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

&nbsp;
## 6.6 Calculando rewards

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F14_raschka.webp" width=400>

- Os rewards são simplesmente rewards de correção, parecidos com os do capítulo 3
- No entanto, há também um format reward implícito: o reward de 1.0 só é concedido se a resposta final estiver no formato `\boxed{}` (via `fallback=None`)

In [11]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(
        answer_text, fallback=None  # Require \boxed{}
    )
    if not extracted:
        return 0.0
    correct = grade_answer(extracted, ground_truth)
    return float(correct)

In [12]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]
rollout_rewards = []

for answer in rollouts:
    reward = reward_rlvr(answer_text=answer, ground_truth="83")
    print(f"Answer: {answer!r}")
    print(f"Reward: {reward}\n")
    rollout_rewards.append(reward)

Answer: '\\boxed{83}'
Reward: 1.0

Answer: 'The correct answer is \\boxed{83}'
Reward: 1.0

Answer: 'The final answer is 83'
Reward: 0.0

Answer: 'We get \\boxed{38}'
Reward: 0.0



- Nota: a equipe do DeepSeek-R1 tentou usar process reward models para pontuar passos intermediários da solução ao treinar o modelo
- No entanto, essas tentativas não tiveram sucesso, e os pesquisadores concluíram que é melhor treinar apenas com rewards de correção da resposta final, sem rewards intermediários

&nbsp;
## 6.7 Preparando sinais de aprendizado a partir dos rollouts via advantages

- O "GR" (group relative) em GRPO se refere ao fato de que o GRPO gera várias respostas (rollouts) por prompt e as compara entre si para construir um sinal de aprendizado

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F15_raschka.webp" width=400>

- A fórmula é bem simples:

$$\text{advantages}_i = \frac{r_i - \mu_r}{\sigma_r + \epsilon}$$

- Aqui, $r_i$ denota o reward do $i$-ésimo rollout, $\mu_r$ é o reward médio do grupo de rollouts, $\sigma_r$ é o desvio padrão correspondente e $\epsilon$ é uma constante pequena, adicionada para estabilidade numérica, para evitar erros de divisão por zero

In [13]:
rewards = torch.tensor(rollout_rewards, device=device)
print(rewards)

tensor([1., 1., 0., 0.])


In [14]:
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([ 0.8659,  0.8659, -0.8659, -0.8659])


- Note que, se todos os rewards de um grupo forem idênticos, por exemplo, todos 0 ou todos 1, então $r_i - \mu_r = 0$ para todos os $i$ rollouts
- Isso significa que o modelo não é atualizado se todas as respostas estiverem corretas ou todas estiverem incorretas

&nbsp;
## 6.8 Pontuando rollouts com log-probabilidades de sequência

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F16_raschka.webp" width=400>

- No capítulo anterior, implementamos uma função `avg_logprob_answer` que calcula log-probabilidades por token para os tokens da resposta
- Essas log-probabilidades médias também costumam ser chamadas de log-probabilidades no nível do token e são comumente usadas para pontuar respostas de LLMs
- Essa média é preferida para pontuação, porque fornece normalização por comprimento
- Matematicamente, isso pode ser expresso como $\frac{1}{T} \sum_{t=1}^{T} \log p_W(y_t \mid y_{<t}, x)$
- Aqui, $y_1, ..., y_T$ denotam os tokens da resposta gerada, de comprimento $T, y_{<t}$ representa todos os tokens gerados anteriormente, $x$ é o prompt de entrada e $W$ denota os parâmetros de peso do modelo
- Essa expressão é matematicamente idêntica à usada no capítulo anterior; simplesmente trocamos $x$ por $y$ para distinguir os tokens de saída gerados do prompt de entrada, por clareza
- Para referência, a função está copiada abaixo

In [15]:
## Chapter 5

@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):

    # Encode prompt and answer tokens separately to get the prompt length later
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)

    # Same as in calc_next_token_logprobas before
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)

    # Index range for positions corresponding to answer tokens
    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1

    # Same as before, except for using start and end
    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Average over the answer token scores
    return torch.mean(next_token_logps).item()

In [16]:
avg_logprob_val = avg_logprob_answer(
                   model, tokenizer, 
                   prompt=prompt,
                   answer=answer_text,
                   device=device) 
print(avg_logprob_val)

-0.061279296875


- No entanto, o GRPO usa log-probabilidades no nível da sequência, não médias no nível do token normalizadas por comprimento, como acima
- Médias no nível do token são úteis para pontuação, já que tornam comparáveis saídas de comprimentos diferentes
- No GRPO, cada rollout recebe um reward e um advantage para a sequência inteira e, para escalar o gradiente corretamente, as log-probabilidades precisam refletir a verossimilhança da sequência completa, o que se obtém somando as log-probabilidades no nível do token
- Caso contrário, tirar a média das log-probabilidades reescalaria implicitamente o sinal de aprendizado pelo comprimento da sequência e distorceria as atualizações da policy, especialmente para rollouts mais longos

- Podemos convertê-la em uma log-probabilidade no nível da sequência abandonando a média e trocando o `torch.mean(next_token_logps)` por um `torch.sum(next_token_logps)`
- Retroativamente, também poderíamos multiplicar o resultado médio pelo número de tokens da resposta para obter o valor sem média

In [17]:
sequence_logprob_val = avg_logprob_val * (len(tokenizer.encode(answer_text)))
print(sequence_logprob_val)

-16.239013671875


- Esses logprobs no nível da sequência escalam linearmente com o comprimento da sequência T
- Isso significa que respostas mais longas sempre recebem logprobs mais negativos
- Isso, por sua vez, incentiva a preferir, entre duas respostas igualmente boas, a mais curta (que é mais barata)
- Logprobs somados incentivam o modelo a parar mais cedo

- Então, como mencionado acima, podemos substituir `torch.mean` por `torch.sum` na função acima
- No entanto, como rodamos a função em modo de inferência no capítulo anterior, usando o decorador `@torch.inference_mode()`, precisamos redefini-la de qualquer forma, já que queremos que o PyTorch rastreie e calcule gradientes
- Além disso, como o `sample_response` da seção 6.5 retorna `token_ids` e `prompt_len`, podemos eliminar a codificação e o cálculo de `full_ids` em `avg_logprob_answer`, para simplificar

In [18]:
def sequence_logprob_draft(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)

    # Positions whose next-token probabilities we want
    # These correspond to predicting token_ids[t + 1] from position t
    start = prompt_len - 1
    end = token_ids.shape[0] - 1

    t_idx = torch.arange(start, end, device=token_ids.device)
    next_tokens = token_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Sum log-probabilities over the answer tokens
    return torch.sum(next_token_logps)

print(sequence_logprob_draft(model, token_ids, prompt_len))

tensor(-16.2998, grad_fn=<SumBackward0>)


- Note que não usamos `.item()` em `torch.sum(next_token_logps)`, para que o PyTorch retorne um tensor (em vez de um float do Python), o que é importante para o cálculo do gradiente
- Como podemos ver, o valor resultante (-16.2998) é quase idêntico ao que obtivemos antes ao reescalar o `avg_logprob_val` pelo número de tokens da resposta (-16.2390); as pequenas diferenças podem ser atribuídas ao comportamento de arredondamento de ponto flutuante

- Abaixo, vamos reescrever a função usando torch.gather, que é um pouco mais idiomático em PyTorch e um pouco mais bem otimizado para GPUs
- No entanto, ambas as funções são matematicamente equivalentes

In [19]:
def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)
    selected = logprobs[:-1].gather(
        1, token_ids[1:].unsqueeze(-1)
    ).squeeze(-1)
    return torch.sum(selected[prompt_len - 1:])

print(sequence_logprob(model, token_ids, prompt_len))

tensor(-16.2998, grad_fn=<SumBackward0>)


In [20]:
rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",
    r"We get \boxed{38}",
]

rollout_logps = []

for text in rollouts:
    token_ids = tokenizer.encode(prompt + " " + text)
    logprob = sequence_logprob(
        model=model,
        token_ids=torch.tensor(token_ids, device=device),
        prompt_len=prompt_len,
    )

    print(f"Answer:  {text}")
    print(f"Logprob: {logprob.item():.4f}\n")

    rollout_logps.append(logprob)

Answer:  \boxed{83}
Logprob: -7.9243

Answer:  The correct answer is \boxed{83}
Logprob: -20.1546

Answer:  The final answer is 83
Logprob: -16.6130

Answer:  We get \boxed{38}
Logprob: -23.3677



- A tendência aqui é que respostas mais curtas e concisas recebem log-probabilidades no nível da sequência mais altas (menos negativas)
- E o menor score é atribuído à única resposta que contém um valor incorreto (38 em vez de 83)
- No geral, log-probabilidades somadas favorecem saídas concisas e corretas

&nbsp;
## 6.9 Dos advantages às atualizações de policy via a loss do GRPO

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F17_raschka.webp" width=400>

In [21]:
logps = torch.stack(rollout_logps)
print(logps)

tensor([ -7.9243, -20.1546, -16.6130, -23.3677], grad_fn=<StackBackward0>)


In [22]:
pg_loss = -(advantages.detach() * logps).mean()
print(pg_loss)

tensor(-2.5764, grad_fn=<NegBackward0>)


- Precisamos do `.detach()` porque queremos tratar os `advantages` como sinais de aprendizado fixos; assim, garantimos que só retropropagamos pelos logprobs
- Precisamos do sinal negativo porque os otimizadores do PyTorch minimizam por padrão, e aqui queremos maximizar os advantages ponderados por logprob

- Em notação matemática, podemos escrever a loss de policy gradient da seguinte forma:

$$\mathcal{L}_{\mathrm{PG}}
= -\frac{1}{N} \sum_{i=1}^{N} A_i \sum_{t=1}^{T_i} \log p_W\!\left( y_t^{(i)} \mid y_{<t}^{(i)}, x^{(i)} \right)$$

- $N$ denota o número de rollouts no batch
- $y_1^{(i)}, ..., y_{T_i}^{(i)}$ são os tokens da $i$-ésima resposta gerada, de comprimento $T_i$
- $y_{<t}^{(i)}$ representa todos os tokens gerados anteriormente naquela resposta
- $x^{(i)}$ é o prompt de entrada correspondente ao $i$-ésimo rollout
- $p_W$ denota a policy do modelo, ou seja, a distribuição de probabilidade sobre os próximos tokens, parametrizada pelos pesos $W$
- $A_i$ é o advantage atribuído ao $i$-ésimo rollout completo
- A soma interna calcula a log-probabilidade no nível da sequência de um rollout
- A média externa calcula as log-probabilidades ponderadas por advantage entre os rollouts

&nbsp;
## 6.10 Juntando tudo em um step do GRPO

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F18_raschka.webp" width=400>

In [23]:
def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
):
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        # Stage 1: generate rollouts
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        # Stage 2: compute rewards
        reward = reward_rlvr(text, example["answer"])
        
        # Stage 4: compute logprobs
        logp = sequence_logprob(model, token_ids, prompt_len)

        roll_logps.append(logp)
        roll_rewards.append(reward)
        samples.append(
            {
                "text": text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        model.train()

    # Stage 2: collect all rewards
    rewards = torch.tensor(roll_rewards, device=device)

    # Stage 3: compute advantages
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # Stage 4: collect all logprobs
    logps = torch.stack(roll_logps)

    # Stage 5: compute policy gradient loss
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss  # In the next chapter we add a KL term here

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }

- Os estágios nos comentários do código correspondem aos estágios da figura do GRPO
- Note que, depois do estágio 1, temos os estágios 2 e 4 (em vez de 3 e 4) nos comentários do código, já que isso resulta em uma implementação mais simples (para não precisarmos implementar vários laços for)

In [24]:
torch.manual_seed(123)

stats = compute_grpo_loss(
    model=model,
    tokenizer=tokenizer,
    example=math_train[4],
    device=device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9
)

pprint(stats)

{'advantages': [0.0, 0.0],
 'loss': -0.0,
 'loss_tensor': tensor(-0., grad_fn=<NegBackward0>),
 'pg_loss': -0.0,
 'rewards': [0.0, 0.0],
 'samples': [{'gen_len': 4, 'reward': 0.0, 'text': ' 14<|endoftext|>'},
             {'gen_len': 256,
              'reward': 0.0,
              'text': ' 4\n'
                      '\n'
                      "To solve the problem, let's break it down step by "
                      'step:\n'
                      '\n'
                      '1. **Define Variables:**\n'
                      '   - Let \\( x \\) be the number of days Sam works.\n'
                      '   - Then, the number of days he does not work is \\( '
                      '20 - x \\).\n'
                      '\n'
                      '2. **Set Up the Earnings Equation:**\n'
                      '   - For each day he works, he earns \\$60.\n'
                      '   - For each day he does not work, he loses \\$30.\n'
                      '   - His total earnings are \\$660.

&nbsp;
## 6.11 Implementando o loop de treinamento do GRPO

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F19_raschka.webp" width=600>

- Pulamos o batching devido aos requisitos de recursos já elevados

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F20_raschka.webp" width=400>

In [25]:
import time

def train_rlvr_grpo(
    model,
    tokenizer,
    math_data,
    device,
    steps=None,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=50,
    checkpoint_dir=".",
    csv_log_path=None,

):
    if steps is None:
        steps = len(math_data)

    # Stage 1: initialize optimizer
    # (the model was already initialized outside the function)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    model.train()
    current_step = 0
    if csv_log_path is None:
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        csv_log_path = f"train_rlvr_grpo_metrics_{timestamp}.csv"
    csv_log_path = Path(csv_log_path)

    try:
        # Stage 2: Iterate over training steps
        for step in range(steps):

            # Stage 3: Reset loss gradient
            # (it's best practice to do this at the beginning of each step)
            optimizer.zero_grad()

            current_step = step + 1
            example = math_data[step % len(math_data)]

            # Stage 4: calculate GRPO loss
            stats = compute_grpo_loss(
                model=model,
                tokenizer=tokenizer,
                example=example,
                device=device,
                num_rollouts=num_rollouts,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )

            # Stage 5: Backward pass to calculate loss gradients
            stats["loss_tensor"].backward()

            # Clip large gradients to improve training stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            # Stage 6: Update model weights using loss gradients
            optimizer.step()

            # Stage 7: Collect rewards, response lengths, and losses
            reward_avg = torch.tensor(stats["rewards"]).mean().item()
            step_tokens = sum(
                sample["gen_len"] for sample in stats["samples"]
            )
            avg_response_len = (
                step_tokens / len(stats["samples"]) 
                if stats["samples"] else 0.0
            )
            append_csv_metrics(
                csv_log_path, current_step, steps, stats["loss"],
                reward_avg, avg_response_len,
            )

            # Print step metrics
            print(
                f"[Step {current_step}/{steps}] "
                f"loss={stats['loss']:.4f} "
                f"reward_avg={reward_avg:.3f} "
                f"avg_resp_len={avg_response_len:.1f}"
            )

            # Sample outputs (every 10 steps) to check if model
            # generates coherent text
            if current_step % 10 == 0:
                print(f"[Step {current_step}] sample outputs")
                for i, sample in enumerate(stats["samples"][:3]):
                    text = sample["text"].replace("\n", "\\n")
                    print(
                        f"  {i+1}) reward={sample['reward']:.3f} "
                        f"len={sample['gen_len']}: {text}"
                    )
                print()

            # Stage 8: Save model checkpoint
            if checkpoint_every and current_step % checkpoint_every == 0:
                ckpt_path = save_checkpoint(
                    model=model,
                    checkpoint_dir=checkpoint_dir,
                    step=current_step,
                )
                print(f"Saved checkpoint to {ckpt_path}")

    # Save a model checkpoint if we interrupt the training early
    except KeyboardInterrupt:
        ckpt_path = save_checkpoint(
            model=model,
            checkpoint_dir=checkpoint_dir,
            step=max(1, current_step),
            suffix="interrupt",
        )
        print(f"\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")
        return model

    return model


def save_checkpoint(model, checkpoint_dir, step, suffix=""):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"-{suffix}" if suffix else ""
    ckpt_path = (
        checkpoint_dir /
        f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth"
    )
    torch.save(model.state_dict(), ckpt_path)
    return ckpt_path


def append_csv_metrics(
    csv_log_path,
    step_idx,
    total_steps,
    loss,
    reward_avg,
    avg_response_len,
):
    if not csv_log_path.exists():
        csv_log_path.write_text(
            "step,total_steps,loss,reward_avg,avg_response_len\n",
            encoding="utf-8",
        )
    with csv_log_path.open("a", encoding="utf-8") as f:
        f.write(
            f"{step_idx},{total_steps},{loss:.6f},{reward_avg:.6f},"
            f"{avg_response_len:.6f}\n"
        )

- Tudo, exceto o estágio 4, o cálculo da loss do GRPO, faz parte do loop de treinamento padrão para treinar redes neurais profundas (incluindo LLMs)
- O `append_csv_metrics` registra os resultados em um arquivo CSV para fins de registro (e para visualizar os resultados no capítulo 7)
- Para uma introdução geral ao treinamento de redes neurais em PyTorch, veja as seções 3 a 8 do meu artigo [PyTorch in One Hour: From Tensors to Training Neural Networks on Multiple GPUs](https://sebastianraschka.com/teaching/pytorch-1h/)

In [26]:
device = get_device()
model.to(device)

torch.manual_seed(0)

train_rlvr_grpo(
    model=model,
    tokenizer=tokenizer,
    math_data=math_train,
    device=device,
    steps=50,
    num_rollouts=4,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
    lr=1e-5,
    checkpoint_every=5,
    checkpoint_dir=".",
    csv_log_path="train_rlvr_grpo_metrics.csv",
)

Using Apple Silicon GPU (MPS)
[Step 1/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=88.0
[Step 2/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=7.5
[Step 3/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=6.5
[Step 4/50] loss=0.0909 reward_avg=0.250 avg_resp_len=6.5
[Step 5/50] loss=1.1001 reward_avg=0.500 avg_resp_len=300.5
Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00005.pth

KeyboardInterrupt. Saved checkpoint to qwen3-0.6B-rlvr-grpo-step00006-interrupt.pth


Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

- Se você tiver problemas de memória ao rodar o código acima, pode reduzir o número de rollouts (por exemplo, `num_rollouts=2`) e o número de tokens por rollout (por exemplo, `max_new_tokens=128`)
- No entanto, para obter um modelo relativamente bom, são necessários pelo menos `num_rollouts=8` e `max_new_tokens=512`
- Se você não conseguir rodar no hardware disponível, sem problemas: a próxima seção mostra como baixar um checkpoint pré-treinado

- Note que, de todo modo, o código provavelmente rodará bem devagar, porque o GRPO é um procedimento intensivo em recursos
- Você pode interromper a execução a qualquer momento, e ela salvará o checkpoint mais recente do modelo na pasta `checkpoints`
- Se você tem interesse em usar GPUs na nuvem, veja o documento [Recursos de GPU na nuvem](../../ch02/02_setup-tips/gpu-instructions.md) para recomendações

- Note que este código não suporta treinamento em batch
- Essa é uma escolha deliberada para manter o código mais simples e legível, e porque amostrar múltiplos rollouts (potencialmente longos) já pode ser bem intensivo em recursos
- No entanto, se você tem acesso a várias GPUs, pode usar a versão opcional deste código, com suporte a batch e múltiplas GPUs, que pode ser encontrada no material suplementar em [../02_rlvr_grpo_scripts_intro](../02_rlvr_grpo_scripts_intro), que treina o modelo mais rápido

&nbsp;
## 6.12 Carregando e avaliando checkpoints salvos do modelo

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch06/CH06_F21_raschka.webp" width=600>

- Os checkpoints salvos podem ser carregados usando o model.load_state_dict(torch.load(model_path)) explicado no capítulo 2, onde model_path referencia o arquivo ".pth" do checkpoint
- Esses arquivos de checkpoint também são compatíveis com os utilitários de avaliação de modelo do capítulo 3
- Por conveniência, você pode usar os scripts de avaliação fornecidos no material complementar do capítulo 3:

```python
uv run ../../ch03/02_math500-verifier-scripts/evaluate_math500.py \
--dataset_size 500 \
--which_model base \
--checkpoint_path checkpoints/qwen3-0.6B-rlvr-grpo-step00050.pth
```


- Se você preferir não rodar o treinamento do GRPO no seu computador porque demora demais, também pode baixar os checkpoints que subi para [rasbt/qwen3-from-scratch-grpo-checkpoints/tree/main/grpo_original_no_kl](https://huggingface.co/rasbt/qwen3-from-scratch-grpo-checkpoints/tree/main/grpo_original_no_kl) (clique no arquivo de checkpoint que você quer baixar e depois clique no botão de [download](https://huggingface.co/rasbt/qwen3-from-scratch-grpo-checkpoints/resolve/main/grpo_original_no_kl/qwen3-0.6B-rlvr-grpo-step00050.pth?download=true))
- Por conveniência, você também pode baixar o checkpoint diretamente aqui, usando Python

In [27]:
from reasoning_from_scratch.qwen3 import download_qwen3_grpo_checkpoints

download_qwen3_grpo_checkpoints(grpo_type="no_kl", step="00050")

✓ qwen3-0.6B-rlvr-grpo-step00050.pth already up-to-date


|      | Método                                 | Step | Máx. de tokens | Nº de rollouts | Acurácia MATH-500 | Média de tokens |
| ---- | -------------------------------------- | ---- | ---------- | ------------ | ------------ | --------------- |
| 1    | Base (capítulo 3)                      | -    |            |              | 15,2%        | 78,85           |
| 2    | Reasoning (capítulo 3)                 | -    |            |              | 48,2%        | 1369,79         |
| 3    | GRPO original, mas sem KL (este capítulo) | 50   | 512        | 8            | 47,4%        | 586,11          |

- Com base na tabela acima, vemos que, após apenas 50 steps, o modelo treinado (linha 3), que é inicializado a partir do modelo base (linha 1), fica quase tão bom quanto a variante de raciocínio original (linha 2)
- Note que treinar por mais tempo pode não melhorar o modelo e até piorá-lo, já que o GRPO pode ser relativamente instável; o próximo capítulo apresenta truques adicionais para melhorar o algoritmo GRPO

&nbsp;
## 6.13 Resumo

- Sem código nesta seção